This DeBERTa-V3-Large model was trained using:
* The original 200 training set;
* The 1000 STEM-1K dataset by [leonidkulyk](https://www.kaggle.com/leonidkulyk); and
* The extra 6500 (500+6000) dataset by [radek1](https://www.kaggle.com/radek1).

In [ ]:
import pandas as pd
import numpy as np
import torch

from datasets import Dataset
from dataclasses import dataclass
from transformers import AutoTokenizer
from transformers.tokenization_utils_base import PreTrainedTokenizerBase, PaddingStrategy
from transformers import AutoModelForMultipleChoice, TrainingArguments, Trainer
from typing import Optional, Union

In [ ]:
options = 'ABCDE'
indices = list(range(5))
option_to_index = {option: index for option, index in zip(options, indices)}
index_to_option = {index: option for option, index in zip(options, indices)}

def preprocess(example):
    
    first_sentence = [example['prompt']] * 5
    second_sentence = []
    for option in options:
        second_sentence.append(example[option])
    tokenized_example = tokenizer(first_sentence, second_sentence, truncation=True)
    tokenized_example['label'] = option_to_index[example['answer']]
    
    return tokenized_example

@dataclass
class DataCollatorForMultipleChoice:
    
    tokenizer: PreTrainedTokenizerBase
    padding: Union[bool, str, PaddingStrategy] = True
    max_length: Optional[int] = None
    pad_to_multiple_of: Optional[int] = None
    
    def __call__(self, features):
        
        label_name = "label" if 'label' in features[0].keys() else 'labels'
        labels = [feature.pop(label_name) for feature in features]
        batch_size = len(features)
        num_choices = len(features[0]['input_ids'])
        flattened_features = [
            [{k: v[i] for k, v in feature.items()} for i in range(num_choices)] for feature in features
        ]
        flattened_features = sum(flattened_features, [])
        
        batch = self.tokenizer.pad(
            flattened_features,
            padding=self.padding,
            max_length=self.max_length,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors='pt',
        )
        batch = {k: v.view(batch_size, num_choices, -1) for k, v in batch.items()}
        batch['labels'] = torch.tensor(labels, dtype=torch.int64)
        
        return batch

def predictions_to_map_output(predictions):
    
    sorted_answer_indices = np.argsort(-predictions)
    top_answer_indices = sorted_answer_indices[:,:3] # Get the first three answers in each row
    top_answers = np.vectorize(index_to_option.get)(top_answer_indices)
    
    return np.apply_along_axis(lambda row: ' '.join(row), 1, top_answers)
#==================================================================================================================
# training_args = TrainingArguments(
#     output_dir='./',
#     evaluation_strategy="no",
#     save_strategy="epoch",
#     load_best_model_at_end=False,
#     learning_rate=3e-5,
#     per_device_train_batch_size=1,
#     per_device_eval_batch_size=1,
#     num_train_epochs=1,
#     weight_decay=0.01,
#     report_to='none'
# )


training_args = TrainingArguments(
    warmup_ratio=0.8,
    learning_rate=5e-6,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=2,
    num_train_epochs=20,
    report_to='none',
    output_dir='.',
    save_strategy='no'
)
#===================================================================================================================
#test_df = pd.read_csv('/kaggle/input/kaggle-llm-science-exam/test.csv')
test_df_all = pd.read_csv('/kaggle/input/more-data/train_3000.csv')
test_df=test_df_all
#test_df['answer'] = 'A'
test_ds = Dataset.from_pandas(test_df)

In [ ]:
#load another 2000 training_set
train_df=test_df_all
train_df=Dataset.from_pandas(train_df)

In [ ]:
model_dir = '/kaggle/input/2023kagglellm-deberta-v3-large-model1'
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForMultipleChoice.from_pretrained(model_dir)
tokenized_dataset=train_df.map(preprocess, batched=False, remove_columns=['prompt', 'A', 'B', 'C', 'D', 'E', 'answer'])
trainer = Trainer(
    model=model,
    args=training_args,
    tokenizer=tokenizer,
    data_collator=DataCollatorForMultipleChoice(tokenizer=tokenizer),
    train_dataset=tokenized_dataset
)

trainer.train()
model.eval()

model.save_pretrained('models/1')



#====================================================
tokenized_test_ds = test_ds.map(preprocess, batched=False, remove_columns=['prompt', 'A', 'B', 'C', 'D', 'E', 'answer'])
test_predictions = trainer.predict(tokenized_test_ds)
submission_df = test_df[['id']]
submission_df['prediction'] = predictions_to_map_output(test_predictions.predictions)
submission_df.to_csv('submission.csv', index=False)
print("Done.")


In [ ]:
def score(prediction,answer,num):
 sum=0
 for i in range(2000,3251):
        if prediction['prediction'][i][0]==answer['answer'][i]:
            sum=sum+1
        if prediction['prediction'][i][2]==answer['answer'][i]:
            sum=sum+1/2
        if prediction['prediction'][i][4]==answer['answer'][i]:
            sum=sum+1/3
 return sum/num
#print(score(subm
score(submission_df,test_df,1251) #  #0.673(original)

In [ ]:
test_df = pd.read_csv('/kaggle/input/kaggle-llm-science-exam/test.csv')
test_df['answer'] = 'A'
test_ds = Dataset.from_pandas(test_df)
tokenized_test_ds = test_ds.map(preprocess, batched=False, remove_columns=['prompt', 'A', 'B', 'C', 'D', 'E', 'answer'])
test_predictions = trainer.predict(tokenized_test_ds)
submission_df = test_df[['id']]
submission_df['prediction'] = predictions_to_map_output(test_predictions.predictions)
submission_df.to_csv('submission.csv', index=False)
print("Done.")